# Bài giảng: Train LightGBM cho Time-Series Forecasting

Notebook này giải thích cho người mới cách pipeline train model chính LightGBM:
1. Tạo feature store (lag, rolling, promo/event, seasonality)
2. Tune siêu tham số bằng expanding-window CV
3. Chọn mô hình theo MAE, đồng thời báo cáo RMSE/R2
4. Tạo `submission.csv`

In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd

In [ ]:
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.feature_store import create_daily_feature_store
from src.forecasting import train_time_series_model

In [ ]:
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
print('RAW_DIR      :', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

## Bước 1 - Tạo feature store daily

Feature store tổng hợp từ nhiều bảng và tạo ra các nhóm feature:
- **Lag**: `revenue_lag_1/7/14/28`, `cogs_lag_1/7/14/28`, ...
- **Rolling**: `revenue_roll_mean_7`, `revenue_roll_std_14`, ...
- **Promo/Event**: `promo_active_count`, `promo_is_active`, ...
- **Seasonality**: `day_of_week`, `month`, `quarter`, `is_weekend`, ...

In [ ]:
daily_features = create_daily_feature_store(processed_dir=PROCESSED_DIR)
print('daily_features shape:', daily_features.shape)
display(daily_features.head(3))

In [ ]:
lag_cols = [c for c in daily_features.columns if '_lag_' in c]
roll_cols = [c for c in daily_features.columns if '_roll_' in c]
promo_cols = [c for c in daily_features.columns if c.startswith('promo_')]
season_cols = [
    c for c in ['day_of_week', 'week_of_year', 'month', 'quarter', 'year', 'is_weekend', 'is_month_start', 'is_month_end']
    if c in daily_features.columns
]

print('Lag features      :', len(lag_cols))
print('Rolling features  :', len(roll_cols))
print('Promo/Event cols  :', len(promo_cols))
print('Seasonality cols  :', len(season_cols))
print('\nVí dụ lag:', lag_cols[:8])
print('Ví dụ rolling:', roll_cols[:8])
print('Ví dụ promo:', promo_cols)

## Bước 2 - Train & tune LightGBM với Time-Series CV

In [ ]:
result = train_time_series_model(
    processed_dir=PROCESSED_DIR,
    raw_dir=RAW_DIR,
    output_dir=PROCESSED_DIR,
    method='lightgbm',
    cv_folds=5,
    cv_valid_size=30,
    cv_min_train_size=180,
    lgbm_max_trials=6,
)

print('Selected method:', result['selected_method'])
print('Selected metrics:', result['selected_metrics'])
print('Submission path:', result['submission_path'])

In [ ]:
best_trial = result['cv']['results']['lightgbm']['best_trial']
print('Best trial params:')
print(json.dumps(best_trial['params'], indent=2))

print('\nBest trial metrics:')
print(json.dumps(best_trial['avg_metrics'], indent=2))

In [ ]:
summary_rows = []
for model_name in ['linear_regression', 'seasonal_naive', 'lightgbm']:
    metrics = result['cv']['results'][model_name]['avg_metrics']
    summary_rows.append({'model': model_name, **metrics})

summary_df = pd.DataFrame(summary_rows).sort_values('mae')
display(summary_df)

## Kết quả đầu ra

- `data/processed/submission.csv`
- `data/processed/baseline_metrics.json`
- `data/processed/daily_feature_store.csv`

Bạn có thể nộp `submission.csv` để test leaderboard, còn `baseline_metrics.json` để phân tích tuning.